In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import hashlib
import secrets
import numpy as np
import matplotlib.pyplot as plt

from ecc import *

---

# Module 4: From ElGamal to ECC

**Where we are:** We can add points, multiply them by scalars, and compress
public keys. Before we jump to signatures (Module 5), we need to understand
the scheme ECC was born from — because ECC didn't invent a new idea, it
transplanted an existing one into a harder group.

That story starts with a problem: how do two people agree on a shared secret
over a public channel?

You might ask: why do we need a *shared* secret if we have public keys? Can't
Bob just encrypt directly to Alice's public key? He can — and ElGamal encryption
(section 4.2) does exactly that. But under the hood, "encrypting to a public key"
*is* establishing a shared secret. When Bob picks a random $k$ and computes
$A^k$ using Alice's public key, he's creating a one-time shared secret that only
Alice can reconstruct (using her private key on Bob's ephemeral $C_1 = g^k$).
Diffie-Hellman is the primitive that makes this possible. Understanding it first
makes everything else — ElGamal encryption, ECDH, even signatures — just
variations on the same trick.

## 4.1 Diffie-Hellman Key Exchange (1976)

Imagine Alice and Bob want to communicate securely, but everyone can see their
messages. They can't just send a password — an eavesdropper would read it.

Diffie and Hellman's idea: use a one-way function so that both parties can
independently arrive at the **same secret**, without ever transmitting it.

**Setup:** Everyone agrees on a large prime $p$ and a generator $g$ — a number
whose powers $g^1, g^2, g^3, \ldots \bmod p$ cycle through all values from
$1$ to $p-1$.

**The exchange:**

```
           Alice                          Bob
           ─────                          ───
  1. Pick secret a                 1. Pick secret b
  2. Compute A = g^a mod p         2. Compute B = g^b mod p
  3. Send A to Bob ──────────────► 3. Receive A
  4. Receive B ◄────────────────── 4. Send B to Alice
  5. Compute B^a = g^(ba) mod p    5. Compute A^b = g^(ab) mod p
           │                              │
           └──── same shared secret ──────┘
                    g^(ab) mod p
```

**Why it works:** Alice computes $B^a = (g^b)^a = g^{ab}$. Bob computes
$A^b = (g^a)^b = g^{ab}$. Same answer — commutativity of exponents.

**Why it's secure:** Nobody can recover anyone else's private key — not even
the participants. Alice knows $a$ and sees $B = g^b$, but she can't find $b$.
Bob knows $b$ and sees $A = g^a$, but he can't find $a$. And the eavesdropper
sees both $A$ and $B$ but can't find $a$ or $b$, and can't compute $g^{ab}$
without one of them. Extracting an exponent from $g^a$ is the discrete log
problem — infeasible for large $p$. The only thing shared is the result
$g^{ab}$, and only Alice and Bob can compute it (each using their own secret
on the other's public value).

**Concrete example with small numbers ($p = 23$, $g = 5$):**

| | Alice | Bob |
|---|---|---|
| Secret | $a = 6$ | $b = 15$ |
| Public | $A = 5^6 \bmod 23 = 8$ | $B = 5^{15} \bmod 23 = 19$ |
| Shared secret | $19^6 \bmod 23 = 2$ | $8^{15} \bmod 23 = 2$ |

The eavesdropper sees 8 and 19 but can't figure out that the shared secret is 2
without knowing 6 or 15.

## 4.2 ElGamal — from key exchange to encryption

In 1985, Taher ElGamal extended Diffie-Hellman into a full encryption scheme.

In DH, Alice and Bob both participate — they each contribute a secret and
arrive at a shared result. But what if Bob wants to send Alice a message and
she's not online to do an exchange? ElGamal solves this: Bob uses Alice's
**published public key** to create a one-time shared secret *by himself*, then
uses it to encrypt.

Think of it as **one-sided Diffie-Hellman**: Bob does the exchange alone,
using Alice's public key as her half.

**Setup:** Same as DH — prime $p$, generator $g$. Alice publishes her public
key $A = g^a \bmod p$ (where $a$ is her secret).

### Encryption (Bob → Alice)

```
  Bob wants to send message m to Alice.
  He knows: p, g, and Alice's public key A = g^a

  Step 1: Pick a random k (fresh, one-time — like a nonce)

  Step 2: Compute C₁ = g^k mod p
           This is Bob's ephemeral "public key" — like his half
           of a DH exchange, but thrown away after one use.

  Step 3: Compute the shared secret S = A^k mod p
           Bob is doing Alice's side of DH for her:
           S = A^k = (g^a)^k = g^(ak)
           Only Alice will be able to recompute this.

  Step 4: Encrypt the message: C₂ = m · S mod p
           The message is now "masked" by the shared secret.

  Step 5: Send (C₁, C₂) to Alice.
```

### Decryption (Alice)

```
  Alice receives (C₁, C₂). She knows her secret a.

  Step 1: Recompute the shared secret S = C₁^a mod p
           Why does this work?
           C₁^a = (g^k)^a = g^(ka) = g^(ak) = (g^a)^k = A^k = S  ✓
           Same S that Bob computed — commutativity of exponents.

  Step 2: Decrypt: m = C₂ · S⁻¹ mod p
           C₂ · S⁻¹ = (m · S) · S⁻¹ = m  ✓
           The shared secret cancels out, revealing the message.
```

### Why nobody else can decrypt

An eavesdropper sees $C_1 = g^k$ and $C_2 = m \cdot S$. To get $S$, she'd need
to compute $(g^k)^a$ — but she doesn't know $a$ (Alice's secret). She could try
to find $k$ from $C_1 = g^k$, but that's the discrete log again. Without $a$ or
$k$, the message stays hidden.

### The problem with ElGamal on integers

The scheme works, but the discrete log in integers mod $p$ can be attacked with
**index calculus** — a subexponential algorithm. To stay safe, you need enormous
keys: 3072 bits for 128-bit security. That's slow, wasteful, and impractical
for things like Bitcoin transactions.

## 4.3 ECC — same scheme, harder group

Koblitz and Miller's insight (1985): **run the exact same schemes, but replace
exponentiation mod $p$ with scalar multiplication on an elliptic curve.**

Everything maps directly:

| | ElGamal (integers mod $p$) | ECC (elliptic curve) |
|---|---|---|
| Public parameter | prime $p$, generator $g$ | curve, generator point $G$ |
| Private key | secret exponent $a$ | secret scalar $d$ |
| Public key | $A = g^a \bmod p$ | $P = d \times G$ |
| Hard problem | given $A$, find $a$ | given $P$, find $d$ |
| Key exchange | $g^{ab}$ | $d_A \cdot d_B \cdot G$ |
| Best attack | subexponential (index calculus) | exponential (Pollard's rho) |

The operations look different (exponentiation vs point multiplication) but the
algebraic structure is identical — both are Abelian groups where the forward
operation is easy and the reverse is hard. That's why the Abelian properties
mattered — they're what allow the same scheme to work in both settings.

The critical difference: there's no known subexponential attack against the
elliptic curve discrete log. The best attack (Pollard's rho) is $O(\sqrt{N})$,
which is **fully exponential**. Same security, a fraction of the key size:

| Security Level | RSA/ElGamal Key | ECC Key | Ratio |
|---------------|----------------|---------|-------|
| 80-bit | 1024 bits | 160 bits | 6.4× |
| 128-bit | 3072 bits | 256 bits | 12× |
| 256-bit | 15360 bits | 512 bits | 30× |

The code below demonstrates both schemes side-by-side — ElGamal with integers,
then the same thing with elliptic curve points.

In [ ]:
# Side-by-side: ElGamal vs ECC key exchange

print("=" * 60)
print("ElGamal Key Exchange (integers mod p)")
print("=" * 60)

# Small ElGamal example (from the essay)
p_eg = 101
alpha = 7  # Primitive root mod 101

# Alice
a_priv = 23  # Alice's private key
beta_a = pow(alpha, a_priv, p_eg)  # Alice's public key
print(f"Alice: private a={a_priv}, public β = α^a mod p = 7^{a_priv} mod 101 = {beta_a}")

# Bob
b_priv = 37  # Bob's private key
beta_b = pow(alpha, b_priv, p_eg)  # Bob's public key
print(f"Bob:   private b={b_priv}, public β'= α^b mod p = 7^{b_priv} mod 101 = {beta_b}")

# Shared secret
shared_eg_a = pow(beta_b, a_priv, p_eg)  # Alice computes β'^a
shared_eg_b = pow(beta_a, b_priv, p_eg)  # Bob computes β^b
print(f"Shared secret: Alice={shared_eg_a}, Bob={shared_eg_b}, Match={shared_eg_a == shared_eg_b}")
print(f"(Both compute α^(ab) mod p = 7^{a_priv*b_priv} mod 101 = {pow(alpha, a_priv*b_priv, p_eg)})")

print(f"\n{'=' * 60}")
print("ECC Key Exchange (ECDH on secp256k1)")
print("=" * 60)

# Alice
alice_priv = secrets.randbelow(SECP_N - 1) + 1
alice_pub = scalar_mult(alice_priv, G)  # Alice's public key = a × G
print(f"Alice: private a (256-bit random), public A = a×G")

# Bob
bob_priv = secrets.randbelow(SECP_N - 1) + 1
bob_pub = scalar_mult(bob_priv, G)  # Bob's public key = b × G
print(f"Bob:   private b (256-bit random), public B = b×G")

# Shared secret: both compute the same point
shared_ecc_a = scalar_mult(alice_priv, bob_pub)   # a × (b×G) = ab×G
shared_ecc_b = scalar_mult(bob_priv, alice_pub)    # b × (a×G) = ab×G
print(f"Shared secret point: {shared_ecc_a == shared_ecc_b}  ✓")
print(f"Both compute a×b×G (same point, never transmitted)")

print(f"\n{'─' * 60}")
print(f"ElGamal: security from α^a mod p  (needs ~3072-bit p)")
print(f"ECC:     security from k×G         (needs ~256-bit N)")
print(f"Same security, 12× smaller keys.")

## 4.2 ECC Encryption / Decryption (ElGamal on Curves)

The essay demonstrates ECC encryption using the curve $y^2 = x^3 - x + 4$ over $\mathbb{F}_{457}$.
The scheme maps directly from ElGamal:

| ElGamal | ECC Analog |
|---------|------------|
| $\beta = \alpha^a \bmod p$ | $Q = d \times G$ (public key) |
| Encrypt: $(\alpha^k, m \cdot \beta^k)$ | Encrypt: $(k \times G, \; P_m + k \times Q)$ |
| Decrypt: $m = t \cdot (\beta')^{-a}$ | Decrypt: $P_m = C_2 - d \times C_1$ |

Multiplication in ElGamal becomes **point addition** in ECC.  
Exponentiation becomes **scalar multiplication**.

In [ ]:
from ecc.small_curve import (
    SmallPoint, small_add, small_mult, small_negate, small_sqrt,
    encode_char_to_point, decode_point_to_char,
    P_SMALL, A_SMALL, B_SMALL, G_SMALL, K_ENC,
)

assert (8**2) % P_SMALL == (4**3 + A_SMALL*4 + B_SMALL) % P_SMALL, "G not on curve!"

d = 101
Q = small_mult(d, G_SMALL)
print(f"=== ECC Encryption (Essay Example) ===")
print(f"Curve: y\u00b2 = x\u00b3 - x + 4 over F_457")
print(f"G = {G_SMALL}")
print(f"Private key d = {d}")
print(f"Public key Q = d\u00d7G = {Q}")

msg_val = 7
pm = encode_char_to_point(msg_val)
print(f"\nMessage: 'H' (value {msg_val}) \u2192 point {pm}")

k_rand = 41
C1 = small_mult(k_rand, G_SMALL)
C2 = small_add(pm, small_mult(k_rand, Q))
print(f"\nEncrypt with random k={k_rand}:")
print(f"  C1 = k\u00d7G = {C1}")
print(f"  C2 = Pm + k\u00d7Q = {C2}")

dC1 = small_mult(d, C1)
pm_recovered = small_add(C2, small_negate(dC1))
msg_recovered = decode_point_to_char(pm_recovered)
print(f"\nDecrypt:")
print(f"  d\u00d7C1 = {dC1}")
print(f"  Pm = C2 - d\u00d7C1 = {pm_recovered}")
print(f"  Decoded: value {msg_recovered} \u2192 '{chr(65 + msg_recovered)}'")
print(f"  Match: {pm == pm_recovered}  \u2713")

### Why decryption works

$$C_2 - d \times C_1 = (P_m + k \times Q) - d \times (k \times G)$$
$$= P_m + k \times (d \times G) - d \times (k \times G)$$
$$= P_m + k \cdot d \times G - d \cdot k \times G$$
$$= P_m \quad \checkmark$$

The random factor $k$ cancels out because both parties have access to the
shared secret $k \cdot d \times G$ through different paths.

---